In [ ]:
pip install tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tldextract]


In [ ]:
import re, math
import tldextract
from urllib.parse import urlparse

IP_PATTERN = re.compile(r"^(\d{1,3}\.){3}\d{1,3}$")
SUSPICIOUS_WORDS = ["login", "verify", "secure", "account", "update", "bank",
                    "signin", "confirm", "password", "paypal", "webscr"]

FEATURE_NAMES = [
    "url_length", "hostname_length", "path_length", "query_length",
    "num_dots", "num_hyphens", "num_underscores", "num_digits", "digit_ratio",
    "has_at", "is_ip", "path_depth", "num_subdomains", "num_params",
    "suspicious_words", "entropy", "has_double_slash_in_path",
    "tld_length", "num_special_chars", "is_safe_tld"
]

def _entropy(s):
    if not s:
        return 0.0
    return -sum((s.count(c) / len(s)) * math.log2(s.count(c) / len(s)) for c in set(s))

def normalize(url):
    url = str(url).strip()
    url = re.sub(r"^[a-zA-Z]+://", "", url)
    url = re.sub(r"^www\d*\.", "", url, flags=re.I)
    return url

def extract_features(url):
    u = normalize(url)

    # Use tldextract for highly accurate domain parsing (fixes .edu.in issue)
    ext = tldextract.extract(u)
    is_ip = int(bool(IP_PATTERN.match(ext.domain)))

    try:
        p = urlparse("http://" + u)
        host, path, q = p.hostname or "", p.path or "", p.query or ""
    except ValueError:
        m = re.match(r"([^/?#]*)([^?#]*)\??([^#]*)", u)
        host, path, q = m.group(1).lower(), m.group(2), m.group(3)

    digits = sum(c.isdigit() for c in u)
    low = u.lower()

    # Calculate true subdomains (e.g. 'login.manavrachna.edu.in' -> 1 subdomain)
    # If there is no subdomain text, it is 0.
    num_subs = len([s for s in ext.subdomain.split('.') if s]) if ext.subdomain and not is_ip else 0

    # Identify trusted institutional TLDs
    safe_tlds = {"edu.in", "ac.in", "gov.in", "nic.in", "edu", "gov"}
    is_safe_tld = int(ext.suffix in safe_tlds)

    # Normalize TLD length so .edu.in is not penalized for being long
    tld_len = len(ext.suffix) if not is_ip else 0
    if ext.suffix.endswith(".in") and tld_len > 3:
        tld_len -= 3

    return [
        len(u), len(host), len(path), len(q),
        u.count("."), u.count("-"), u.count("_"), digits,
        digits / max(len(u), 1),
        int("@" in u), is_ip, path.count("/"),
        num_subs,
        len(q.split("&")) if q else 0,
        sum(w in low for w in SUSPICIOUS_WORDS),
        _entropy(u), int("//" in path),
        tld_len,
        len(re.findall(r"[^a-zA-Z0-9]", u)),
        is_safe_tld
    ]

In [ ]:
# version 1

import pandas as pd, joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from features import extract_features, FEATURE_NAMES

from tqdm import tqdm

df = pd.read_csv("urldata.csv", usecols=["url", "result"])
df = df.dropna().drop_duplicates(subset="url")
print(df["result"].value_counts())

print("Extracting features (a few minutes for 450k rows)...")
X = pd.DataFrame([extract_features(u) for u in tqdm(df["url"], desc="Extracting")], columns=FEATURE_NAMES)
y = df["result"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(
    n_estimators=300, n_jobs=-1, class_weight="balanced",
    min_samples_leaf=2, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy :", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall   :", round(recall_score(y_test, pred), 4))
print("F1       :", round(f1_score(y_test, pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["benign", "malicious"]))
print(pd.Series(model.feature_importances_, index=FEATURE_NAMES)
        .sort_values(ascending=False).head(10))

joblib.dump(model, "phishing_model_version1.pkl")

result
0    345738
1    104438
Name: count, dtype: int64
Extracting features (a few minutes for 450k rows)...


Extracting:  24%|██▍       | 109075/450176 [00:05<00:16, 21067.76it/s]


KeyboardInterrupt: 

In [ ]:
# version 2

import json
import os
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split

from features import FEATURE_NAMES, extract_features, normalize

DATA = "urldata.csv"
TRANCO = Path("top-1m.csv")      # optional: rank,domain  (no header)
AUG_FROM_DATASET = 20000          # benign bare domains taken from benign training URLs
AUG_FROM_TRANCO = 20000           # sampled from the Tranco file, if present
MIN_INDIAN = 3000                 # ensure many .in domains if Tranco has them
SEED = 42
SAMPLE = int(os.environ.get("SAMPLE", 0))   # set SAMPLE=60000 for a quick test run

# Legitimate sites you care about; printed after training as a sanity check
PROBE = ["manavrachna.edu.in", "mru.edu.in", "iitd.ac.in", "iitb.ac.in", "ugc.gov.in",
         "isro.gov.in", "irctc.co.in", "sbi.co.in", "amazon.in", "flipkart.com",
         "google.com", "wikipedia.org", "github.com", "mit.edu", "bbc.co.uk"]


def is_probe(host: str) -> bool:
    """True for the probe domains (and their subdomains). They are kept OUT of training so the
    legitimate-domain check below stays an honest test of unseen sites."""
    return any(host == p or host.endswith("." + p) for p in PROBE)


def host_of(url: str) -> str:
    return normalize(url).split("/")[0].split("?")[0].lower()


# ---------------------------------------------------------------- load + split first
df = pd.read_csv(DATA, usecols=["url", "result"]).dropna().drop_duplicates(subset="url")
if SAMPLE:
    df = df.sample(SAMPLE, random_state=SEED)
print("Rows:", len(df), "| malicious share:", round(df["result"].mean(), 3))

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["result"])

# ---------------------------------------------------------------- augment TRAIN only
rng = np.random.default_rng(SEED)
pat = re.compile(r"^[a-z0-9.-]+\.[a-z]{2,}$")

ben_hosts = pd.Series([host_of(u) for u in train_df.loc[train_df["result"] == 0, "url"]]).drop_duplicates()
ben_hosts = ben_hosts[ben_hosts.map(lambda h: bool(pat.match(h)) and not is_probe(h))]
aug = list(ben_hosts.sample(min(AUG_FROM_DATASET, len(ben_hosts)), random_state=SEED))
print("Added bare domains from benign training URLs:", len(aug))

tranco_hold = []   # legitimate domains the model never sees: an honest false-alarm test
if TRANCO.exists():
    tr = pd.read_csv(TRANCO, header=None, names=["rank", "domain"]).head(300000)
    tr["domain"] = tr["domain"].astype(str).str.lower()
    tr = tr[tr["domain"].map(lambda h: bool(pat.match(h)))]
    n_probe = int(tr["domain"].map(is_probe).sum())
    tr = tr[~tr["domain"].map(is_probe)]
    print(f"Probe domains found in the Tranco file and kept OUT of training: {n_probe}")
    indian = tr[tr["domain"].str.endswith(".in")]
    rest = tr[~tr["domain"].str.endswith(".in")]
    pick = pd.concat([
        indian.sample(min(MIN_INDIAN, len(indian)), random_state=SEED),
        rest.sample(min(AUG_FROM_TRANCO - min(MIN_INDIAN, len(indian)), len(rest)), random_state=SEED),
    ])["domain"].tolist()
    rng.shuffle(pick)
    n_hold = max(1, int(0.10 * len(pick)))
    tranco_hold, pick = pick[:n_hold], pick[n_hold:]       # 10% never used for training
    aug += pick
    print(f"Added Tranco domains: {len(pick)} for training, {len(tranco_hold)} held out for testing")
else:
    print("No top-1m.csv found. Using dataset-derived domains only (see header for the Tranco option).")

forms = ["{d}", "https://{d}", "https://www.{d}/", "http://{d}/"]
aug_urls = [forms[i].format(d=d) for d, i in zip(aug, rng.integers(0, len(forms), len(aug)))]
X_urls = pd.concat([train_df["url"], pd.Series(aug_urls)], ignore_index=True)
y_train = np.concatenate([train_df["result"].values, np.zeros(len(aug_urls), dtype=int)])

# ---------------------------------------------------------------- features + model
print("Extracting features...")
X_train = pd.DataFrame([extract_features(u) for u in X_urls], columns=FEATURE_NAMES)
X_test = pd.DataFrame([extract_features(u) for u in test_df["url"]], columns=FEATURE_NAMES)
y_test = test_df["result"].values

model = RandomForestClassifier(n_estimators=300, n_jobs=-1, class_weight="balanced",
                               min_samples_leaf=2, random_state=SEED)
model.fit(X_train, y_train)

# ---------------------------------------------------------------- evaluate (untouched test set)
pred = model.predict(X_test)
print("\nAccuracy :", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall   :", round(recall_score(y_test, pred), 4))
print("F1       :", round(f1_score(y_test, pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["benign", "malicious"]))

# ---------------------------------------------------------------- legitimate-domain sanity check
P = model.predict_proba(pd.DataFrame([extract_features(t) for t in PROBE], columns=FEATURE_NAMES))[:, 1]
print("Legitimate-domain check (malicious probability, lower is better):")
for t, p in zip(PROBE, P):
    print(f"  {p:5.2f}  {'<-- flagged' if p >= 0.5 else ''}  {t}")

metrics = {"accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred),
           "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred)}

# ---------------------------------------------------------------- unseen legitimate domains (Tranco hold-out)
if tranco_hold:
    Xh = pd.DataFrame([extract_features("https://" + d) for d in tranco_hold], columns=FEATURE_NAMES)
    ph = model.predict_proba(Xh)[:, 1]
    is_in = np.array([d.endswith(".in") for d in tranco_hold])
    fa = float((ph >= 0.5).mean())
    print(f"\nFalse-alarm rate on {len(ph)} UNSEEN legitimate Tranco domains: {fa:.1%}")
    if is_in.any():
        print(f"  of which .in domains ({int(is_in.sum())}): {float((ph[is_in] >= 0.5).mean()):.1%}")
    metrics["unseen_legit_false_alarm_rate"] = fa

json.dump(metrics, open("metrics.json", "w"))
joblib.dump(model, "phishing_model.pkl", compress=3)  # ~4x smaller file, same predictions
print("\nSaved phishing_model.pkl and metrics.json")

Rows: 450176 | malicious share: 0.232
Added bare domains from benign training URLs: 20000
No top-1m.csv found. Using dataset-derived domains only (see header for the Tranco option).
Extracting features...

Accuracy : 0.9276
Precision: 0.8331
Recall   : 0.8603
F1       : 0.8464
Confusion matrix:
 [[65547  3601]
 [ 2919 17969]]
              precision    recall  f1-score   support

      benign       0.96      0.95      0.95     69148
   malicious       0.83      0.86      0.85     20888

    accuracy                           0.93     90036
   macro avg       0.90      0.90      0.90     90036
weighted avg       0.93      0.93      0.93     90036

Legitimate-domain check (malicious probability, lower is better):
   0.01    manavrachna.edu.in
   0.01    mru.edu.in
   0.21    iitd.ac.in
   0.21    iitb.ac.in
   0.01    ugc.gov.in
   0.01    isro.gov.in
   0.31    irctc.co.in
   0.25    sbi.co.in
   0.11    amazon.in
   0.31    flipkart.com
   0.13    google.com
   0.15    wikipedia.org
  

Xg boost

In [ ]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

print("Initializing Indian-Centric Phishing URL Feature Extractor & Trainer...")

# 1. Advanced Lexical & Indian Context Feature Extractor
def extract_features(url):
    features = {}
    try:
        parsed = urlparse(str(url))
        netloc = parsed.netloc.lower()
        path = parsed.path.lower()
        full_url = str(url).lower()

        # --- Basic Lexical & Structural Features ---
        features['url_length'] = len(full_url)
        features['domain_length'] = len(netloc)
        features['path_length'] = len(path)

        # IP address check in domain
        features['has_ip'] = 1 if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', netloc) else 0

        # Special character counts
        features['dot_count'] = full_url.count('.')
        features['hyphen_count'] = full_url.count('-')
        features['at_symbol'] = 1 if '@' in full_url else 0
        features['double_slash'] = 1 if '//' in path else 0
        features['question_mark'] = 1 if '?' in full_url else 0
        features['equal_sign'] = 1 if '=' in full_url else 0
        features['underscore_count'] = full_url.count('_')

        # --- Indian Threat-Vector Heuristics ---
        # Country Code Top-Level Domain (ccTLD) checks
        features['is_in_cctld'] = 1 if netloc.endswith('.in') or netloc.endswith('.co.in') else 0
        features['is_co_in'] = 1 if netloc.endswith('.co.in') else 0

        # High-risk regional brand & service impersonation keywords (Banking, UPI, Gov, Delivery)
        indian_keywords = [
            'sbi', 'yono', 'upi', 'paytm', 'phonepe', 'gpay', 'googlepay',
            'hdfc', 'icici', 'axis', 'pnb', 'bob', 'kotak', 'idbi', 'canara',
            'irctc', 'aadhaar', 'uidai', 'income-tax', 'incometax', 'digilocker',
            'ration', 'covid', 'pmkisan', 'fastag', 'upsc', 'cbse', 'neet'
        ]

        # Count keyword occurrences and set boolean flag
        keyword_hits = sum(1 for kw in indian_keywords if kw in full_url)
        features['indian_keyword_count'] = keyword_hits
        features['has_indian_keyword'] = 1 if keyword_hits > 0 else 0

        # Suspicious combination: e.g., brand keyword + non-official TLD or hyphenated domain
        features['suspicious_brand_hyphen'] = 1 if (features['has_indian_keyword'] == 1 and '-' in netloc) else 0

    except Exception as e:
        # Fallback for malformed URLs
        for key in ['url_length', 'domain_length', 'path_length', 'has_ip', 'dot_count',
                    'hyphen_count', 'at_symbol', 'double_slash', 'question_mark', 'equal_sign',
                    'underscore_count', 'is_in_cctld', 'is_co_in', 'indian_keyword_count',
                    'has_indian_keyword', 'suspicious_brand_hyphen']:
            features[key] = 0

    return features

# 2. Dataset Pipeline Function
def process_dataset(csv_path):
    print(f"Loading dataset from {csv_path}...")
    df = pd.read_csv(csv_path)

    # Ensure expected columns exist (adjust names based on your CSV schema if needed)
    if 'url' not in df.columns or 'label' not in df.columns:
        raise ValueError("CSV must contain 'url' and 'label' columns.")

    print("Extracting lexical and regional features...")
    feature_rows = [extract_features(url) for url in df['url']]
    features_df = pd.DataFrame(feature_rows)

    features_df['label'] = df['label'].values
    return features_df

# --- Execution Example ---
if __name__ == "__main__":
    # Replace with your downloaded dataset CSV filename from Kaggle
    dataset_filename = "phishing_url_dataset_unique.csv"

    try:
        processed_data = process_dataset(dataset_filename)

        X = processed_data.drop(columns=['label'])
        y = processed_data['label']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

        print(f"Training XGBoost model on {len(X_train)} samples...")
        model = xgb.XGBClassifier(
            n_estimators=150,
            learning_rate=0.08,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )

        model.fit(X_train, y_train)

        # Evaluation
        preds = model.predict(X_test)
        print("\n--- Model Performance Report ---")
        print(f"Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")
        print("\nClassification Report:")
        print(classification_report(y_test, preds))

        # Save model and feature list for your Streamlit/Vercel deployment
        joblib.dump(model, 'indian_phishing_xgb_model.pkl')
        print("\nModel successfully saved as 'indian_phishing_xgb_model.pkl'!")

    except FileNotFoundError:
        print(f"[{dataset_filename}] not found. Place your Kaggle CSV in the working directory and run the script.")

Initializing Indian-Centric Phishing URL Feature Extractor & Trainer...
Loading dataset from phishing_url_dataset_unique.csv...
Extracting lexical and regional features...
Training XGBoost model on 39049 samples...

--- Model Performance Report ---
Accuracy: 99.97%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4882
           1       1.00      1.00      1.00      4881

    accuracy                           1.00      9763
   macro avg       1.00      1.00      1.00      9763
weighted avg       1.00      1.00      1.00      9763


Model successfully saved as 'indian_phishing_xgb_model.pkl'!


In [ ]:
import joblib
import pandas as pd
from urllib.parse import urlparse
import re

# 1. Official Whitelist (Hard bypass for genuine root domains)
TRUSTED_DOMAINS = {
    "onlinesbi.sbi", "sbicard.com", "hdfcbank.com", "icicibank.com",
    "irctc.co.in", "uidai.gov.in", "gov.in", "nic.in", "axisbank.com"
}

def is_whitelisted(url):
    try:
        parsed = urlparse(str(url))
        netloc = parsed.netloc.lower()
        # Check if domain or its parent matches any trusted domain
        for trusted in TRUSTED_DOMAINS:
            if netloc == trusted or netloc.endswith("." + trusted):
                return True
    except Exception:
        pass
    return False

# 2. Match Feature Extractor to Model's Training Schema
def extract_features(url):
    features = {}
    try:
        parsed = urlparse(str(url))
        netloc = parsed.netloc.lower()
        path = parsed.path.lower()
        full_url = str(url).lower()

        # --- Basic Lexical & Structural Features ---
        features['url_length'] = len(full_url)
        features['domain_length'] = len(netloc)
        features['path_length'] = len(path)

        # IP address check in domain
        features['has_ip'] = 1 if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', netloc) else 0

        # Special character counts
        features['dot_count'] = full_url.count('.')
        features['hyphen_count'] = full_url.count('-')
        features['at_symbol'] = 1 if '@' in full_url else 0
        features['double_slash'] = 1 if '//' in path else 0
        features['question_mark'] = 1 if '?' in full_url else 0
        features['equal_sign'] = 1 if '=' in full_url else 0
        features['underscore_count'] = full_url.count('_')

        # --- Indian Threat-Vector Heuristics ---
        # Country Code Top-Level Domain (ccTLD) checks
        features['is_in_cctld'] = 1 if netloc.endswith('.in') or netloc.endswith('.co.in') else 0
        features['is_co_in'] = 1 if netloc.endswith('.co.in') else 0

        # High-risk regional brand & service impersonation keywords (Banking, UPI, Gov, Delivery)
        indian_keywords = [
            'sbi', 'yono', 'upi', 'paytm', 'phonepe', 'gpay', 'googlepay',
            'hdfc', 'icici', 'axis', 'pnb', 'bob', 'kotak', 'idbi', 'canara',
            'irctc', 'aadhaar', 'uidai', 'income-tax', 'incometax', 'digilocker',
            'ration', 'covid', 'pmkisan', 'fastag', 'upsc', 'cbse', 'neet'
        ]

        # Count keyword occurrences and set boolean flag
        keyword_hits = sum(1 for kw in indian_keywords if kw in full_url)
        features['indian_keyword_count'] = keyword_hits
        features['has_indian_keyword'] = 1 if keyword_hits > 0 else 0

        # Suspicious combination: e.g., brand keyword + non-official TLD or hyphenated domain
        features['suspicious_brand_hyphen'] = 1 if (features['has_indian_keyword'] == 1 and '-' in netloc) else 0

    except Exception as e:
        # Fallback for malformed URLs
        for key in ['url_length', 'domain_length', 'path_length', 'has_ip', 'dot_count',
                    'hyphen_count', 'at_symbol', 'double_slash', 'question_mark', 'equal_sign',
                    'underscore_count', 'is_in_cctld', 'is_co_in', 'indian_keyword_count',
                    'has_indian_keyword', 'suspicious_brand_hyphen']:
            features[key] = 0

    return features

# 3. Robust Inference Function with Whitelist Check
def classify_url(url, model):
    # Step 1: Check Whitelist First
    if is_whitelisted(url):
        return "✅ LEGITIMATE (Verified Official Domain)", 100.0

    # Step 2: ML Model Inference
    feats = extract_features(url)
    df_input = pd.DataFrame([feats])

    prediction = model.predict(df_input)[0]
    probability = model.predict_proba(df_input)[0]

    label_text = "🚨 PHISHING / MALICIOUS" if prediction == 1 else "✅ LEGITIMATE"
    confidence = probability[prediction] * 100
    return label_text, confidence

# --- Test Suite ---
if __name__ == "__main__":
    try:
        model = joblib.load('indian_phishing_xgb_model.pkl')
    except FileNotFoundError:
        print("Model file not found. Please ensure 'indian_phishing_xgb_model.pkl' is in your directory.")
        exit()

    test_urls = [
        "https://www.onlinesbi.sbi/",
        "https://www.irctc.co.in/",
        "https://www.hdfcbank.com/",
        "https://uidai.gov.in/",
        "http://sbi-yono-update-kyc.in/login/secure",
        "https://upi-rewards-claim-bonus.co.in/verify",
        "https://www.apollohospitals.com/hospitals/apollo-hospitals-delhi"
    ]

    print("--- Running Updated Inference Test ---")
    for url in test_urls:
        result, conf = classify_url(url, model)
        print(f"\nURL: {url}")
        print(f"Result: {result} (Confidence: {conf:.2f}%)")

--- Running Updated Inference Test ---

URL: https://www.onlinesbi.sbi/
Result: ✅ LEGITIMATE (Verified Official Domain) (Confidence: 100.00%)

URL: https://www.irctc.co.in/
Result: ✅ LEGITIMATE (Verified Official Domain) (Confidence: 100.00%)

URL: https://www.hdfcbank.com/
Result: ✅ LEGITIMATE (Verified Official Domain) (Confidence: 100.00%)

URL: https://uidai.gov.in/
Result: ✅ LEGITIMATE (Verified Official Domain) (Confidence: 100.00%)

URL: http://sbi-yono-update-kyc.in/login/secure
Result: 🚨 PHISHING / MALICIOUS (Confidence: 99.97%)

URL: https://upi-rewards-claim-bonus.co.in/verify
Result: 🚨 PHISHING / MALICIOUS (Confidence: 99.98%)

URL: https://www.apollohospitals.com/hospitals/apollo-hospitals-delhi
Result: 🚨 PHISHING / MALICIOUS (Confidence: 99.99%)
